# Heteroscedastic Feature Prediction for Graph Generation

This notebook documents the motivation and implementation of heteroscedastic feature prediction in our BiGG-based graph generator. This technique addresses the **feature under-dispersion** problem, where the deterministic feature decoder collapses generated features towards the conditional mean, producing synthetic graphs with unrealistically compressed feature distributions.

**Overview:**
1. The problem: deterministic decoders and mean regression
2. Heteroscedastic regression as a solution
3. The Gaussian NLL loss and its relationship to MSE
4. Implementation in BiGG's feature extension
5. Practical considerations: variance clamping and training stability
6. Initial results and the feature memorization problem
7. Usage

## 1. The Problem: Mean Regression in Feature Decoding

In our BiGG extension, node features are predicted from the LSTM hidden state $\mathbf{h}$ via an MLP decoder:

$$\hat{\mathbf{x}} = f_\theta(\mathbf{h})$$

This decoder is trained with **mean squared error** (MSE), which minimises the expected squared distance between the prediction and the target:

$$\mathcal{L}_{\text{MSE}} = \sum_{i=1}^{d} (x_i - \hat{x}_i)^2$$

The MSE-optimal prediction is the **conditional mean** $\mathbb{E}[\mathbf{x} | \mathbf{h}]$ (Bishop, 2006, Section 1.5.5). This is a well-known property: MSE regression always converges to the mean of the target distribution conditioned on the input. The decoder has no mechanism to express uncertainty or variance — it outputs a single point estimate.

At generation time, this produces feature vectors that are clustered tightly around the conditional mean for each hidden state. Since many nodes share similar hidden state trajectories (especially in the autoregressive LSTM), the result is a synthetic graph whose feature distribution is **compressed** relative to the original.

```
Original features:  std ≈ 1.0 per feature (after z-score normalisation)
Synthetic features: std ≈ 0.2 - 0.35 per feature
```

The means are roughly correct (bias < 0.1), but the spread is missing. This is not a training failure — the model is doing exactly what MSE asks it to do. The problem is that MSE is the wrong objective for a generative model that needs to reproduce a *distribution*, not just predict a mean.

## 2. Solution: Heteroscedastic Regression

Instead of predicting a single point $\hat{\mathbf{x}}$, we have the network predict the parameters of a **Gaussian distribution** for each feature:

$$\hat{\boldsymbol{\mu}}, \hat{\boldsymbol{\sigma}}^2 = g_\theta(\mathbf{h})$$

$$p(\mathbf{x} | \mathbf{h}) = \prod_{i=1}^{d} \mathcal{N}(x_i; \hat{\mu}_i, \hat{\sigma}_i^2)$$

This is called **heteroscedastic** regression because the predicted variance $\hat{\sigma}_i^2$ can vary per input $\mathbf{h}$ and per feature dimension $i$ (Nix & Weigend, 1994). The model learns not just *what* value to predict, but *how uncertain* it is about that prediction.

At generation time, we **sample** from the predicted distribution using the reparameterisation trick (Kingma & Welling, 2014):

$$\mathbf{x} = \hat{\boldsymbol{\mu}} + \hat{\boldsymbol{\sigma}} \odot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, I)$$

This injects variance into the generated features in a principled way: features that are tightly determined by the graph structure will have low predicted $\hat{\sigma}^2$, while features with high natural variability will have high predicted $\hat{\sigma}^2$.

### Why not just add fixed noise at generation time?

A simpler alternative would be to keep the MSE decoder and add post-hoc Gaussian noise with a fixed standard deviation $\sigma$. This has two drawbacks:

1. **Homoscedastic**: the same $\sigma$ applies to all features and all nodes. In reality, some features may have high natural variance while others are nearly deterministic.
2. **Requires calibration**: $\sigma$ must be estimated externally (e.g., from training residuals), adding a separate step. The heteroscedastic approach learns it end-to-end.

## 3. The Gaussian NLL Loss

The network is trained by maximising the log-likelihood of the training data under the predicted Gaussian. For a single node with feature vector $\mathbf{x}$ and predicted parameters $\hat{\boldsymbol{\mu}}, \hat{\boldsymbol{\sigma}}^2$:

$$\log p(\mathbf{x} | \mathbf{h}) = -\frac{1}{2} \sum_{i=1}^{d} \left[ \log \hat{\sigma}_i^2 + \frac{(x_i - \hat{\mu}_i)^2}{\hat{\sigma}_i^2} \right] + \text{const}$$

Equivalently, the **negative log-likelihood** (NLL) loss to minimise is:

$$\mathcal{L}_{\text{NLL}} = \frac{1}{2} \sum_{i=1}^{d} \left[ \log \hat{\sigma}_i^2 + \frac{(x_i - \hat{\mu}_i)^2}{\hat{\sigma}_i^2} \right]$$

In practice, we predict the **log-variance** $\hat{v}_i = \log \hat{\sigma}_i^2$ instead of the variance directly, which is numerically more stable and avoids the need for a positivity constraint:

$$\mathcal{L}_{\text{NLL}} = \frac{1}{2} \sum_{i=1}^{d} \left[ \hat{v}_i + \frac{(x_i - \hat{\mu}_i)^2}{\exp(\hat{v}_i)} \right]$$

### Relationship to MSE

If the predicted log-variance is constant ($\hat{v}_i = c$ for all $i$), the gradient of $\mathcal{L}_{\text{NLL}}$ with respect to $\hat{\mu}$ reduces to a scaled version of the MSE gradient. In this sense, Gaussian NLL is a **strict generalisation** of MSE — the model can always recover MSE behaviour by learning a constant variance, but it also has the freedom to learn input-dependent variance when the data warrants it (Kendall & Gal, 2017).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Demonstrate that Gaussian NLL generalises MSE

torch.manual_seed(42)
target = torch.randn(100, 8)  # 100 samples, 8 features
pred_mean = target + 0.1 * torch.randn_like(target)  # slightly noisy predictions

# MSE loss
mse = F.mse_loss(pred_mean, target, reduction='sum')

# Gaussian NLL with constant log-variance = 0 (i.e., sigma^2 = 1)
log_var = torch.zeros_like(target)
nll = 0.5 * torch.sum(log_var + (target - pred_mean) ** 2 / torch.exp(log_var))

print(f"MSE loss:                    {mse.item():.4f}")
print(f"Gaussian NLL (log_var=0):    {nll.item():.4f}")
print(f"Ratio (NLL / MSE):           {nll.item() / mse.item():.4f}")
print(f"\nWith constant log_var=0, NLL = 0.5 * MSE (plus a constant).")
print(f"The gradients w.r.t. the mean are proportional.")

## 4. Implementation

The heteroscedastic feature prediction is implemented as an optional flag (`hetero_feat`) in both `BiggWithFeatsAndLabels` and `BiggWithConditionedFeats`. When enabled:

1. The output MLP (`nodefeat_pred`) produces $2d$ values instead of $d$: the first $d$ are the predicted means $\hat{\boldsymbol{\mu}}$, and the last $d$ are the log-variances $\hat{\mathbf{v}}$.
2. During **training**, the loss switches from MSE to Gaussian NLL.
3. During **generation**, features are sampled from $\mathcal{N}(\hat{\boldsymbol{\mu}}, \exp(\hat{\mathbf{v}}))$ via reparameterisation.
4. The **sampled** features (not the raw mean) are embedded and fed back into the LSTM for state update, ensuring the recurrent state reflects the actual generated values.

When disabled (default), all codepaths are identical to the original MSE-based implementation.

In [ ]:
# Simplified illustration of the heteroscedastic predict_node_feats
# (See bigg/extension/customized_models.py for the full implementation)

class HeteroscedasticPredictor(nn.Module):
    def __init__(self, embed_dim, feat_dim, hetero_feat=False):
        super().__init__()
        self.feat_dim = feat_dim
        self.hetero_feat = hetero_feat
        out_dim = 2 * feat_dim if hetero_feat else feat_dim
        self.feat_pred = nn.Linear(embed_dim, out_dim)

    def forward(self, h, target=None):
        raw = self.feat_pred(h)

        if self.hetero_feat:
            mean = raw[:, :self.feat_dim]
            log_var = torch.clamp(raw[:, self.feat_dim:], -4.0, 2.0)
        else:
            mean = raw

        if target is None:
            # Generation: sample from predicted distribution
            if self.hetero_feat:
                std = torch.exp(0.5 * log_var)
                feats = mean + std * torch.randn_like(mean)
            else:
                feats = mean
            return feats, 0
        else:
            # Training: compute loss
            if self.hetero_feat:
                loss = 0.5 * (log_var + (target - mean) ** 2 / torch.exp(log_var))
            else:
                loss = (target - mean) ** 2
            return mean, loss.sum() / self.feat_dim


# Compare generation-time behaviour
torch.manual_seed(42)
embed_dim, feat_dim = 64, 8
h = torch.randn(1, embed_dim)

det_model = HeteroscedasticPredictor(embed_dim, feat_dim, hetero_feat=False)
het_model = HeteroscedasticPredictor(embed_dim, feat_dim, hetero_feat=True)

# Copy weights so the mean predictions are comparable
with torch.no_grad():
    het_model.feat_pred.weight[:feat_dim] = det_model.feat_pred.weight
    het_model.feat_pred.bias[:feat_dim] = det_model.feat_pred.bias
    # Initialise log-variance head to predict log_var ≈ -1 (moderate variance)
    het_model.feat_pred.bias[feat_dim:] = -1.0

det_model.eval()
het_model.eval()

n_samples = 500
det_samples = torch.stack([det_model(h)[0] for _ in range(n_samples)]).squeeze(1)
het_samples = torch.stack([het_model(h)[0] for _ in range(n_samples)]).squeeze(1)

print(f"Deterministic (MSE) — std across samples: {det_samples.std(dim=0).mean():.4f}")
print(f"Heteroscedastic     — std across samples: {het_samples.std(dim=0).mean():.4f}")
print(f"\nThe deterministic model produces identical features every time.")
print(f"The heteroscedastic model samples from a learned distribution.")

## 5. Practical Considerations

### Log-variance clamping

Unconstrained log-variance can diverge during training. If the model finds it difficult to predict the mean accurately, it can trivially reduce the NLL by predicting very large $\hat{v}_i$ — effectively saying "I'm very uncertain" to avoid being penalised for wrong means. This is a well-documented failure mode in heteroscedastic regression (Stirn et al., 2023).

We clamp the predicted log-variance to $[-4, 2]$:

$$\hat{v}_i = \text{clamp}(\hat{v}_i^{\text{raw}}, -4, 2)$$

| Log-variance | Variance $\sigma^2$ | Std $\sigma$ | Interpretation |
|:---:|:---:|:---:|:---|
| -4 | 0.018 | 0.135 | Minimum: near-deterministic prediction |
| 0 | 1.0 | 1.0 | Unit variance (matches z-scored features) |
| 2 | 7.39 | 2.72 | Maximum: high uncertainty |

This range is wide enough for the model to express meaningful variance differences (a 400x range in $\sigma^2$) while preventing degenerate solutions.

### Interaction with existing components

- **Dynamic loss weighting**: The Gaussian NLL replaces MSE within the `ll_cont` term. The existing dynamic normalisation (which scales `w_cont` based on calibration magnitudes) continues to balance continuous, label, and structural losses.
- **Hidden state noise**: The noise injection on $\mathbf{h}$ during training is orthogonal — it regularises the hidden state trajectory, while heteroscedastic prediction addresses the output distribution. Both can be used together, and increasing noise may help prevent feature memorization (see Section 6).
- **Scheduled sampling**: When scheduled sampling is active and the model uses its own predictions for state update, it feeds back the predicted **mean** (not a sample), maintaining training stability. SS directly addresses the train/generate distribution shift but carries risk of destabilising structural learning (see Section 6).

### Motivation in the context of this thesis

The primary goal of this work is to measure utility degradation under varying levels of differential privacy. For that measurement to be meaningful, the **non-private baseline** must produce synthetic graphs whose feature distributions are faithful to the original. If features are already compressed before privacy is applied, the observed utility degradation conflates two effects: generator artifacts and privacy costs. The heteroscedastic output head is designed to eliminate the former, providing a cleaner baseline for privacy experiments.

## 6. Initial Results and the Feature Memorization Problem

### Results on Reddit (300 epochs, normalized, BFS)

The first hetero_feat run used `lw_1_1` (equal loss weights) to maximise signal from the feature loss change. Results on the Reddit dataset compared against three non-hetero baselines with varying loss weights:

| Run | Edges (org: 168,016) | Mean degree (org: 16.3) | Avg clustering (org: 0.00) | Std ratio (target: 1.0) | Mean \|bias\| |
|:---|:---:|:---:|:---:|:---:|:---:|
| `lw_0.01_0.01` (det) | 134,664 | 12.26 | 0.0010 | 0.2856 | 0.0964 |
| `lw_0.1_0.1` (det) | 224,026 | 20.40 | 0.0052 | 0.3486 | 0.0739 |
| `lw_1_1` (det) | 179,090 | 16.30 | 0.0695 | 0.2008 | 0.0671 |
| `lw_1_1` (hetero) | 23,060 | 2.10 | 0.0124 | **0.9104** | 0.1826 |

**Feature dispersion is solved.** The hetero run achieves a std ratio of 0.91 — nearly matching the original distribution's spread — up from 0.20–0.35 in the deterministic runs. The bias is slightly higher (0.18 vs 0.07–0.10) but this is a much smaller problem than the collapsed variance.

**Structure is severely degraded.** The hetero run produces only 23,060 edges (~14% of the original), with a mean degree of 2.1. At least one node has degree 10,965 (near-complete star), and anomaly count is overestimated (509 vs 366 original). The equal loss weights give the feature/label losses too much influence, starving the structural loss of gradient signal.

### Root cause: feature memorization

Training on a single graph means the autoregressive LSTM hidden states $\mathbf{h}$ are deterministic for a given node ordering. The model learns a near-perfect lookup from each node's $\mathbf{h}$ to its features — visible as the per-node MSE (cont loss) approaching zero during training despite `noise_std=0.1`.

At generation time, the autoregressive decisions differ from training, producing hidden states that are similar but not identical. The memorized mapping doesn't transfer well: outputs cluster around the learned conditional means, producing the 0.20–0.35 std ratio in non-hetero runs. The hetero approach fixes this at generation time by sampling from the predicted variance, but the underlying memorization persists.

### Planned next steps

1. **`hetero` + `lw_0.1_0.1` + more epochs**: Down-weight feature/label losses so structure retains gradient signal, while the variance head still provides spread at generation time. Structure loss was still dropping at epoch 300, suggesting more epochs will help.
2. **Increase hidden state noise** (e.g., `noise_std=0.3`): Directly targets memorization by making the $\mathbf{h} \to \mathbf{x}$ mapping noisier during training, forcing the decoder to learn a smoother function that generalises better to unseen hidden states. Low risk — worst case loses some feature accuracy.
3. **Scheduled sampling** (if noise is insufficient): Exposes the model to its own prediction errors during training, directly addressing the train/generate distribution shift. Higher risk — changes hidden states feeding into structure, features, and labels simultaneously, which could destabilise structural learning (as observed with `lw_1_1` hetero). Should be started late in training after structure has converged, with a low max probability.

## 7. Usage

The `hetero_feat` flag is exposed through the full training pipeline:

```bash
# Via SLURM — set HETERO_FEAT=true in train_bigg.slurm
HETERO_FEAT=true

# Via shell script
bash scripts/train/train_bigg.sh reddit -1 1 300 0.001 256 0.1 0.0 0 True zscore 0.01,0.01 true

# Or directly via the pipeline
python -m bigg.extension.pipeline \
  -data_dir reddit \
  -hetero_feat \
  ...
```

The flag affects the save name: generated graphs are saved with a `_hetero` or `_det` suffix to distinguish runs.

When `hetero_feat` is not set (default), all codepaths are identical to the original MSE-based implementation — no retraining of existing models is needed.

## References

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning.* Springer. Section 1.5.5 (loss functions for regression).
- Nix, D. A. & Weigend, A. S. (1994). *Estimating the Mean and Variance of the Target Probability Distribution.* IEEE ICNN. (Original heteroscedastic neural network regression)
- Kingma, D. P. & Welling, M. (2014). *Auto-Encoding Variational Bayes.* ICLR. (Reparameterisation trick)
- Kendall, A. & Gal, Y. (2017). *What Uncertainties Do We Need in Bayesian Deep Learning for Computer Vision?* NeurIPS. (Heteroscedastic aleatoric uncertainty in neural networks)
- Stirn, A., Jeanselme, V. & Knowles, D. A. (2023). *Faithful Heteroscedastic Regression with Neural Networks.* AISTATS. (Variance collapse and clamping strategies)
- Dai, H. et al. (2020). *Scalable Deep Generative Modeling for Sparse Graphs.* ICML. (BiGG)